In [1]:
# 7-2-2026

In [32]:
import glob
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist
from itertools import combinations
from scipy.stats import kendalltau

In [7]:
train_x_dir = "../transfer-matrix/train_X"

# glob for domain csvs
domain_files = glob.glob(os.path.join(train_x_dir, "domain_*.csv"))

In [8]:
domain_files

['../transfer-matrix/train_X\\domain_0.csv',
 '../transfer-matrix/train_X\\domain_1.csv',
 '../transfer-matrix/train_X\\domain_11.csv',
 '../transfer-matrix/train_X\\domain_12.csv',
 '../transfer-matrix/train_X\\domain_13.csv',
 '../transfer-matrix/train_X\\domain_16.csv',
 '../transfer-matrix/train_X\\domain_18.csv',
 '../transfer-matrix/train_X\\domain_19.csv',
 '../transfer-matrix/train_X\\domain_2.csv',
 '../transfer-matrix/train_X\\domain_20.csv',
 '../transfer-matrix/train_X\\domain_21.csv',
 '../transfer-matrix/train_X\\domain_22.csv',
 '../transfer-matrix/train_X\\domain_23.csv',
 '../transfer-matrix/train_X\\domain_25.csv',
 '../transfer-matrix/train_X\\domain_26.csv',
 '../transfer-matrix/train_X\\domain_27.csv',
 '../transfer-matrix/train_X\\domain_28.csv',
 '../transfer-matrix/train_X\\domain_29.csv',
 '../transfer-matrix/train_X\\domain_30.csv',
 '../transfer-matrix/train_X\\domain_32.csv',
 '../transfer-matrix/train_X\\domain_33.csv',
 '../transfer-matrix/train_X\\domain_

In [9]:
base_features = [
    "fwi_mean", "lccs_class_1", "lccs_class_2", "lccs_class_3", "lccs_class_4",
    "lccs_class_6", "lccs_class_7", "ndvi", "pop_dens", "rel_hum", "skt",
    "ssr", "ssrd", "swvl1", "swvl2", "swvl3", "swvl4", "t2m_max", "t2m_mean",
    "t2m_min", "tp", "vpd", "ws10"
]

# fixed seed
SEED = 5
N_SAMPLES = 1000

In [10]:
domain_samples = {}

for f in domain_files:
    # pull domain id out of filename, ex "domain_7.csv": 7
    domain_id = int(os.path.basename(f).replace("domain_", "").replace(".csv", ""))

    df = pd.read_csv(f)

    # keep only the base features, drops target col and anything else automatically
    df = df[base_features]

    # sample 1000 rows, or less if domain somehow has fewer (shouldnt happen, all active domains have 2000+)
    n = min(N_SAMPLES, len(df))
    sample = df.sample(n=n, random_state=SEED)

    domain_samples[domain_id] = sample

In [ ]:
len(domain_samples) # good

34

In [ ]:
# stack all domans together to fit one scaler, not perdomain
#   per domain scaling would hide distribution differences between domains which is
#   exactly what mmd is suposed to detect, one scaler keeps everything on the same ruler
all_rows = pd.concat(domain_samples.values(), axis=0)

scaler = StandardScaler().fit(all_rows)

# apply same scaler to each domain, store back as numpy arrays for mmd step
domain_scaled = {
    domain_id: scaler.transform(sample)
    for domain_id, sample in domain_samples.items()
}

In [ ]:
len(all_rows) # 34x1000, good

34000

In [17]:
# pool a sample across domains to compute bandwidth once, reused for every pair
# doing this per pair would change the "ruler" each time, breaks comparability across matrix
rng = np.random.default_rng(SEED)

pooled_for_bandwidth = np.vstack(list(domain_scaled.values()))

In [18]:
# cap size for compute, dont need every row to get a stable median
max_bw_sample = 5000
if len(pooled_for_bandwidth) > max_bw_sample:
    idx = rng.choice(len(pooled_for_bandwidth), size=max_bw_sample, replace=False)
    pooled_for_bandwidth = pooled_for_bandwidth[idx]

# pairwise euclidean distances, median of these is sigma
dists = pdist(pooled_for_bandwidth, metric="euclidean")
sigma = np.median(dists)

In [ ]:
print(f"bandwidth (sigma): {sigma:.4f}")
#  expected euclidean distance between two random points is about sqrt(2 * 22) = 6.6

bandwidth (sigma): 6.1006


In [23]:
def rbf_kernel_matrix(X, Y, sigma):
    # squared euclidean dist between all rows of X and Y
    sq_dists = np.sum(X**2, axis=1)[:, None] + np.sum(Y**2, axis=1)[None, :] - 2 * X @ Y.T
    return np.exp(-sq_dists / (2 * sigma**2))

def unbiased_mmd2(X, Y, sigma):
    n, m = len(X), len(Y)

    Kxx = rbf_kernel_matrix(X, X, sigma)
    Kyy = rbf_kernel_matrix(Y, Y, sigma)
    Kxy = rbf_kernel_matrix(X, Y, sigma)

    # exclude diagonal terms (i=j) for unbiased estimate
    sum_xx = (Kxx.sum() - np.trace(Kxx)) / (n * (n - 1))
    sum_yy = (Kyy.sum() - np.trace(Kyy)) / (m * (m - 1))
    sum_xy = Kxy.sum() / (n * m)

    return sum_xx + sum_yy - 2 * sum_xy


In [24]:
domain_ids = sorted(domain_scaled.keys())
n_domains = len(domain_ids)
id_to_idx = {d: i for i, d in enumerate(domain_ids)}

mmd_matrix = np.zeros((n_domains, n_domains))


In [25]:
# only compute unique pairs, mmd is symmetric so fill both sides at once
for i, j in combinations(domain_ids, 2):
    m2 = unbiased_mmd2(domain_scaled[i], domain_scaled[j], sigma)
    idx_i, idx_j = id_to_idx[i], id_to_idx[j]
    mmd_matrix[idx_i, idx_j] = m2
    mmd_matrix[idx_j, idx_i] = m2

print(f"computed mmd for {n_domains * (n_domains - 1) // 2} unique pairs")

computed mmd for 561 unique pairs


In [26]:
# negate distance into predicted transferability, same idea as rawdist
predicted_transfer = -mmd_matrix

# build dataframe with domain ids as row/col labels for readability
mmd_df = pd.DataFrame(predicted_transfer, index=domain_ids, columns=domain_ids)

In [27]:
mmd_df.shape

(34, 34)

In [28]:
mmd_df.head()

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-0.091753,-0.295190,-0.220502,-0.046464,-0.210145,-0.177884,-0.069680,-0.116900,-0.199710,...,-0.141869,-0.225514,-0.162635,-0.061955,-0.161644,-0.419346,-0.217384,-0.165180,-0.317172,-0.095015
1,-0.091753,-0.000000,-0.547598,-0.271180,-0.082083,-0.270786,-0.342029,-0.089200,-0.221858,-0.388278,...,-0.355044,-0.391497,-0.325813,-0.041312,-0.300087,-0.655168,-0.438146,-0.163378,-0.494685,-0.192390
2,-0.295190,-0.547598,-0.000000,-0.565534,-0.414482,-0.517862,-0.208324,-0.436485,-0.226869,-0.319881,...,-0.091695,-0.219412,-0.229660,-0.397175,-0.193048,-0.195189,-0.144865,-0.461812,-0.518829,-0.311635
4,-0.220502,-0.271180,-0.565534,-0.000000,-0.252171,-0.015105,-0.335054,-0.295700,-0.282398,-0.436782,...,-0.443404,-0.186991,-0.165344,-0.251971,-0.320255,-0.562626,-0.540233,-0.070253,-0.117704,-0.198213
5,-0.046464,-0.082083,-0.414482,-0.252171,-0.000000,-0.239118,-0.183065,-0.078711,-0.125252,-0.166563,...,-0.222090,-0.303790,-0.212553,-0.056273,-0.168546,-0.494505,-0.257269,-0.197364,-0.366313,-0.159041


In [29]:
mmd_df.to_csv("mmd_transfer_matrix.csv")

In [30]:
mmd_T = pd.read_csv("mmd_transfer_matrix.csv")
mmd_T.set_index("Unnamed: 0", inplace=True)
mmd_T.index.name = None
mmd_T.index = mmd_T.index.astype(int)
mmd_T.columns = mmd_T.columns.astype(int)
mmd_T.head()

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-0.091753,-0.295190,-0.220502,-0.046464,-0.210145,-0.177884,-0.069680,-0.116900,-0.199710,...,-0.141869,-0.225514,-0.162635,-0.061955,-0.161644,-0.419346,-0.217384,-0.165180,-0.317172,-0.095015
1,-0.091753,-0.000000,-0.547598,-0.271180,-0.082083,-0.270786,-0.342029,-0.089200,-0.221858,-0.388278,...,-0.355044,-0.391497,-0.325813,-0.041312,-0.300087,-0.655168,-0.438146,-0.163378,-0.494685,-0.192390
2,-0.295190,-0.547598,-0.000000,-0.565534,-0.414482,-0.517862,-0.208324,-0.436485,-0.226869,-0.319881,...,-0.091695,-0.219412,-0.229660,-0.397175,-0.193048,-0.195189,-0.144865,-0.461812,-0.518829,-0.311635
4,-0.220502,-0.271180,-0.565534,-0.000000,-0.252171,-0.015105,-0.335054,-0.295700,-0.282398,-0.436782,...,-0.443404,-0.186991,-0.165344,-0.251971,-0.320255,-0.562626,-0.540233,-0.070253,-0.117704,-0.198213
5,-0.046464,-0.082083,-0.414482,-0.252171,-0.000000,-0.239118,-0.183065,-0.078711,-0.125252,-0.166563,...,-0.222090,-0.303790,-0.212553,-0.056273,-0.168546,-0.494505,-0.257269,-0.197364,-0.366313,-0.159041


In [34]:
true_matrix = pd.read_csv("../transfer-matrix/rf_transfer_matrix.csv")
true_matrix.set_index("Unnamed: 0", inplace=True)
true_matrix.index.name = None
true_matrix.index = true_matrix.index.astype(int)
true_matrix.columns = true_matrix.columns.astype(int)
true_matrix.head(30)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.392727,0.291660,0.127238,0.058840,0.214817,0.084705,0.149056,0.182358,0.173877,0.026375,...,0.228505,0.243433,0.149609,0.275203,0.112525,-0.011557,0.116862,0.158796,0.025454,0.029928
1,0.142172,0.395760,0.030643,0.010376,0.180707,0.064432,0.056092,0.128808,0.097788,0.039590,...,0.167477,0.032806,0.029704,0.193033,0.055264,0.066550,0.108203,0.199762,0.023240,-0.037710
2,0.196600,0.214445,0.368173,0.023968,0.207323,0.140139,0.172990,0.231019,0.156741,0.057226,...,0.197402,0.223793,0.121457,0.248807,0.134485,0.053503,-0.005610,0.253965,0.031899,0.059566
4,0.235489,0.227173,0.127394,0.410382,0.215954,0.186207,0.125068,0.196603,0.165264,0.034514,...,0.178055,0.245668,0.147579,0.284360,0.217076,-0.003941,0.037643,0.306750,0.137154,0.020968
5,0.190803,0.206530,0.097887,0.087230,0.464604,0.144950,0.159361,0.204884,0.264641,0.104493,...,0.197195,0.232923,0.117228,0.292623,0.170687,-0.020694,0.115293,0.297271,0.067951,0.051770
6,0.178282,0.238235,0.034447,0.220869,0.191487,0.317381,0.118250,0.163631,0.119401,0.021430,...,0.202794,0.247076,0.125801,0.255555,0.201974,-0.068777,0.099243,0.321076,0.102840,-0.017236
7,0.113396,0.054282,0.061550,0.069370,0.044484,0.109329,0.329688,0.092023,0.159883,0.051387,...,0.184071,0.179713,0.142222,0.192045,0.209176,0.035736,0.042035,0.159817,0.008022,0.022467
8,0.181002,0.228547,-0.006209,0.023346,0.133774,0.117255,0.194411,0.424899,0.064289,-0.004919,...,0.093679,0.136592,0.101116,0.173933,0.095452,-0.003312,0.050942,0.240088,0.051530,0.007182
11,0.214508,0.227264,0.109767,0.124790,0.237549,0.076848,0.107019,0.207917,0.551275,-0.003236,...,0.256072,0.166287,0.102690,0.366209,0.181682,-0.007782,0.147501,0.206234,-0.057620,0.064859
12,0.118927,0.175802,-0.016922,0.112255,0.159920,0.070012,0.077276,0.026258,0.103184,0.502334,...,0.163311,-0.091433,0.060118,0.163172,0.119053,0.080449,0.089125,0.118189,0.068853,-0.070408


In [35]:
full_tau, _ = kendalltau(mmd_T.values.flatten(), true_matrix.values.flatten())
full_tau

np.float64(0.22201741210157128)

In [ ]:
mask = ~np.eye(len(mmd_T), dtype=bool)
off_diag_tau, _ = kendalltau(mmd_T.values[mask], true_matrix.values[mask])
print(f"off diagonal kendall tau: {off_diag_tau:.4f}")
# zero self distance always maps to high self transferability, 
#   so it inflates overall without actually proving mmd ranks off diagonal pairs correctly

off diagonal kendall tau: 0.1751


In [ ]:
print(true_matrix.index.equals(mmd_T.index) and true_matrix.columns.equals(mmd_T.columns))
# wow so it's note a mistake :)

True


In [38]:
# a lot of this low correlation can be attributed to how mmd is symmetirc, so it mathematically
#   cannot compare i,j to j,i